In [0]:
%pip install xgboost scikit-learn matplotlib pandas numpy mlflow

# Restart Python after install
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.xgboost
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Tell MLflow to use Unity Catalog as the model registry
mlflow.set_registry_uri("databricks-uc")

# Create or set the MLflow experiment
# This is where all your training runs will be tracked
mlflow.set_experiment("/Users/vvce22cseaiml0106@vvce.ac.in/llm_cost_forecasting")

print("MLflow version:", mlflow.__version__)
print("XGBoost version:", xgb.__version__)
print("Setup complete!")

In [0]:
# Load the gold table into a pandas dataframe
df = spark.table("llm_pulse_dev.gold.daily_cost_by_team").toPandas()

# Sort by date and team
df['event_date'] = pd.to_datetime(df['event_date'])
df = df.sort_values(['team', 'event_date']).reset_index(drop=True)

print(f"Total rows: {len(df)}")
print(f"Date range: {df['event_date'].min()} to {df['event_date'].max()}")
print(f"Teams: {df['team'].unique()}")
print(f"\nSample data:")
df.head(10)

In [0]:
fig, ax = plt.subplots(figsize=(14, 5))

for team in df['team'].unique():
    team_df = df[df['team'] == team]
    ax.plot(team_df['event_date'],
            team_df['rolling_7d_avg_cost'],
            label=team, linewidth=1.5)

ax.set_title('7-day rolling average LLM cost per team over time')
ax.set_xlabel('Date')
ax.set_ylabel('Cost USD')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nCost statistics per team:")
df.groupby('team')['total_cost_usd'].agg(['mean','sum','max']).round(4)

In [0]:
def create_features(df):
    """
    Create time series features for cost forecasting.
    We derive all features from the date and rolling stats —
    no future information is used (no data leakage).
    """
    df = df.copy()

    # ── Date features ──────────────────────────────────────
    df['day_of_week']   = df['event_date'].dt.dayofweek   # 0=Mon, 6=Sun
    df['day_of_month']  = df['event_date'].dt.day
    df['month']         = df['event_date'].dt.month
    df['week_of_year']  = df['event_date'].dt.isocalendar().week.astype(int)
    df['is_weekend']    = (df['day_of_week'] >= 5).astype(int)

    # ── Lag features — past values as predictors ───────────
    # "What did this team spend 1 day ago, 7 days ago, 14 days ago?"
    for lag in [1, 3, 7, 14, 30]:
        df[f'cost_lag_{lag}d'] = (
            df.groupby('team')['total_cost_usd']
              .shift(lag)
        )

    # ── Rolling statistics ─────────────────────────────────
    df['rolling_mean_7d']  = df.groupby('team')['total_cost_usd'].transform(
        lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
    df['rolling_mean_14d'] = df.groupby('team')['total_cost_usd'].transform(
        lambda x: x.shift(1).rolling(14, min_periods=1).mean())
    df['rolling_std_7d']   = df.groupby('team')['total_cost_usd'].transform(
        lambda x: x.shift(1).rolling(7,  min_periods=1).std().fillna(0))

    # ── Team encoding — convert team name to number ────────
    team_map = {'product':0, 'data':1, 'marketing':2, 'ops':3, 'finance':4}
    df['team_encoded'] = df['team'].map(team_map)

    return df

# Apply feature engineering
df_features = create_features(df)

# Drop rows with NaN from lag features
df_features = df_features.dropna()

print(f"Rows after feature engineering: {len(df_features)}")
print(f"\nFeature columns created:")
feature_cols = [
    'day_of_week','day_of_month','month','week_of_year','is_weekend',
    'cost_lag_1d','cost_lag_3d','cost_lag_7d','cost_lag_14d','cost_lag_30d',
    'rolling_mean_7d','rolling_mean_14d','rolling_std_7d',
    'total_calls','rolling_7d_avg_cost','team_encoded'
]
print(feature_cols)

In [0]:
# ── Prepare train / test split ─────────────────────────────
# Use last 14 days as test set — simulate predicting future
X = df_features[feature_cols]
y = df_features['total_cost_usd']

# Time-based split — never random split time series data!
split_date = df_features['event_date'].max() - pd.Timedelta(days=14)
train_mask = df_features['event_date'] <= split_date
test_mask  = df_features['event_date'] >  split_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")

# ── Train XGBoost with MLflow tracking ────────────────────
with mlflow.start_run(run_name="xgboost_cost_forecast_v1"):

    # Define model parameters
    params = {
        'n_estimators':     300,
        'max_depth':        4,
        'learning_rate':    0.05,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'random_state':     42,
        'objective':        'reg:squarederror',
    }

    # Log all parameters to MLflow
    mlflow.log_params(params)
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size",  len(X_test))
    mlflow.log_param("features",   feature_cols)

    # Train the model
    model = xgb.XGBRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    # ── Evaluate ───────────────────────────────────────────
    y_pred = model.predict(X_test)
    y_pred = np.maximum(y_pred, 0)   # cost cannot be negative

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100

    # Log metrics to MLflow
    mlflow.log_metric("MAE",  round(mae,  6))
    mlflow.log_metric("RMSE", round(rmse, 6))
    mlflow.log_metric("MAPE", round(mape, 2))

    # ── Feature importance plot ────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 6))
    xgb.plot_importance(model, ax=ax, max_num_features=15,
                        title='Feature importance — cost forecasting model')
    plt.tight_layout()
    mlflow.log_figure(fig, "feature_importance.png")
    plt.show()

    # ── Log the model to MLflow ────────────────────────────
    input_example = X_train.iloc[:5]
    mlflow.xgboost.log_model(
        model,
        artifact_path = "cost_forecast_model",
        registered_model_name = "llm_pulse_dev.gold.cost_forecast_model",
        input_example = input_example
    )

    print(f"\nModel training complete!")
    print(f"MAE  (mean absolute error):  ${mae:.6f}")
    print(f"RMSE (root mean sq error):   ${rmse:.6f}")
    print(f"MAPE (mean abs % error):     {mape:.2f}%")

In [0]:
from datetime import timedelta

# ── Generate next 30 days of predictions per team ─────────
teams     = df['team'].unique()
last_date = df['event_date'].max()
future_dates = [last_date + timedelta(days=i) for i in range(1, 31)]

predictions = []

for team in teams:
    team_df = df_features[df_features['team'] == team].copy()

    for future_date in future_dates:
        # Build feature row for this future date
        row = {
            'day_of_week':        future_date.dayofweek,
            'day_of_month':       future_date.day,
            'month':              future_date.month,
            'week_of_year':       future_date.isocalendar()[1],
            'is_weekend':         int(future_date.dayofweek >= 5),
            'cost_lag_1d':        team_df['total_cost_usd'].iloc[-1],
            'cost_lag_3d':        team_df['total_cost_usd'].iloc[-3] if len(team_df)>=3  else team_df['total_cost_usd'].mean(),
            'cost_lag_7d':        team_df['total_cost_usd'].iloc[-7] if len(team_df)>=7  else team_df['total_cost_usd'].mean(),
            'cost_lag_14d':       team_df['total_cost_usd'].iloc[-14] if len(team_df)>=14 else team_df['total_cost_usd'].mean(),
            'cost_lag_30d':       team_df['total_cost_usd'].iloc[-30] if len(team_df)>=30 else team_df['total_cost_usd'].mean(),
            'rolling_mean_7d':    team_df['total_cost_usd'].tail(7).mean(),
            'rolling_mean_14d':   team_df['total_cost_usd'].tail(14).mean(),
            'rolling_std_7d':     team_df['total_cost_usd'].tail(7).std(),
            'total_calls':        team_df['total_calls'].tail(7).mean(),
            'rolling_7d_avg_cost':team_df['rolling_7d_avg_cost'].iloc[-1],
            'team_encoded':       {'product':0,'data':1,'marketing':2,'ops':3,'finance':4}[team],
        }

        pred_cost = float(model.predict(pd.DataFrame([row]))[0])
        pred_cost = max(0, pred_cost)  # no negative costs

        predictions.append({
            'prediction_date': future_date.date(),
            'team':            team,
            'predicted_cost_usd': round(pred_cost, 6),
            'prediction_horizon_days': (future_date.date() - last_date.date()).days,
        })

# Convert to Spark DataFrame and save to gold schema
pred_df = pd.DataFrame(predictions)
print(pred_df.head(10))

# Save to Delta table
(spark.createDataFrame(pred_df)
     .write
     .mode("overwrite")
     .saveAsTable("llm_pulse_dev.gold.cost_predictions"))

print(f"\nSaved {len(pred_df)} predictions to gold.cost_predictions")

In [0]:
%sql
-- Check predictions look reasonable
SELECT
    team,
    MIN(predicted_cost_usd)   AS min_predicted,
    MAX(predicted_cost_usd)   AS max_predicted,
    ROUND(AVG(predicted_cost_usd), 4) AS avg_predicted,
    COUNT(*)                  AS days_predicted
FROM llm_pulse_dev.gold.cost_predictions
GROUP BY team
ORDER BY avg_predicted DESC;